In [2]:
import json
import re
from pathlib import Path

In [2]:
src_dir = Path("raw_transcripts")
output_dir = Path("processed")
output_dir.mkdir(exist_ok=True)

src_files = list(src_dir.glob("*.json"))

print("Number of raw files found:", len(src_files))
print("Output folder ready at:", output_dir.resolve())

Number of raw files found: 56
Output folder ready at: C:\Users\Bikram\earningsiq\data\processed


In [ ]:
def find_qa_start(structured_content):
    for i, turn in enumerate(structured_content):
        text_lower = turn["text"].lower()
        if turn["speaker"] == "Operator":
            starts_qa = (
                "first question" in text_lower
                or "take our first" in text_lower
                or "go first to" in text_lower
                or "we will now begin the question" in text_lower
                or "we'll now open" in text_lower
            )
            if starts_qa:
                return i
    
    return int(len(structured_content) * 0.6)


In [ ]:
def split_sections(structured_content):
    qna_start = find_qa_start(structured_content)
    pre_qna = structured_content[:qna_start]
    qna_section = structured_content[qna_start:]
    return pre_qna, qna_section


In [ ]:
def extract_qa_pairs(qa_section):
    pairs = []
    i = 0
    
    while i < len(qa_section):
        turn = qa_section[i]
        
        if turn["speaker"] == "Operator":
            i = i + 1
            continue
        
        question_speaker = turn["speaker"]
        question_text = turn["text"]
        
        answer_parts = []
        j = i + 1
        
        while j < len(qa_section) and qa_section[j]["speaker"] != "Operator":
            if qa_section[j]["speaker"] == question_speaker:
                break
            
            answer_parts.append(qa_section[j]["text"])
            j = j + 1
        
        if answer_parts:
            pairs.append({
                "analyst": question_speaker,
                "question": question_text,
                "answer": " ".join(answer_parts),
                "question_word_count": len(question_text.split()),
                "answer_word_count": len(" ".join(answer_parts).split())
            })
        
        i = j if j > i else i + 1
    
    return pairs



In [ ]:
def extract_numbers(text):
    pattern = r'\$[\d,]+(?:\.\d+)?(?:\s?(?:billion|million|trillion))?|[\d,]+(?:\.\d+)?%'
    matches = re.findall(pattern, text)
    return matches


In [ ]:
def extract_guidance_sentences(text):
    guidance_words = ["we expect", "we anticipate", "we project", "we forecast", "going forward", "outlook", "next quarter", "full year"]
    
    sentences = text.split(". ")
    
    guidance_sentences = []
    for sentence in sentences:
        sentence_lower = sentence.lower()
        for word in guidance_words:
            if word in sentence_lower:
                guidance_sentences.append(sentence.strip())
                break
    
    return guidance_sentences


In [10]:
def process_one_transcript(filepath):
    with open(filepath, encoding="utf-8") as f:
        raw = json.load(f)
    
    structured_content = raw["structured_content"]
    
    prepared, qa_section = split_sections(structured_content)
    qa_pairs = extract_qa_pairs(qa_section)
    
    prepared_text = " ".join(turn["text"] for turn in prepared)
    full_text = raw["content"]
    
    numbers = extract_numbers(full_text)
    guidance = extract_guidance_sentences(prepared_text)
    
    result = {
        "symbol": raw["symbol"],
        "company_name": raw["company_name"],
        "quarter": raw["quarter"],
        "year": raw["year"],
        "date": raw["date"],
        "total_word_count": len(full_text.split()),
        "prepared_remarks": prepared,
        "qa_pairs": qa_pairs,
        "total_qa_pairs": len(qa_pairs),
        "numbers_mentioned": numbers,
        "forward_guidance": guidance
    }
    
    return result

In [10]:
success_count = 0
failed_count = 0
zero_pairs_files = []

for filepath in src_files:
    try:
        result = process_one_transcript(filepath)
        
        output_filename = output_dir / filepath.name
        with open(output_filename, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        print(filepath.name, "- OK |", result["total_word_count"], "words |", result["total_qa_pairs"], "Q&A pairs")
        success_count = success_count + 1
        
        if result["total_qa_pairs"] == 0:
            zero_pairs_files.append(filepath.name)
        
    except Exception as e:
        print(filepath.name, "- FAILED:", str(e))
        failed_count = failed_count + 1

print()
print("Total processed successfully:", success_count)
print("Total failed:", failed_count)
print("Files with 0 Q&A pairs:", zero_pairs_files)

AAPL_Q1_2023.json - OK | 7761 words | 18 Q&A pairs
AAPL_Q1_2024.json - OK | 7709 words | 22 Q&A pairs
AAPL_Q2_2023.json - OK | 8036 words | 22 Q&A pairs
AAPL_Q2_2024.json - OK | 8103 words | 32 Q&A pairs
AAPL_Q3_2023.json - OK | 8061 words | 22 Q&A pairs
AAPL_Q3_2024.json - OK | 7969 words | 28 Q&A pairs
AAPL_Q4_2023.json - OK | 8485 words | 32 Q&A pairs
AAPL_Q4_2024.json - OK | 7926 words | 30 Q&A pairs
AMZN_Q1_2023.json - OK | 7365 words | 5 Q&A pairs
AMZN_Q1_2024.json - OK | 8484 words | 6 Q&A pairs
AMZN_Q2_2023.json - OK | 7220 words | 6 Q&A pairs
AMZN_Q2_2024.json - OK | 7982 words | 7 Q&A pairs
AMZN_Q3_2023.json - OK | 8423 words | 5 Q&A pairs
AMZN_Q3_2024.json - OK | 7736 words | 6 Q&A pairs
AMZN_Q4_2023.json - OK | 8846 words | 6 Q&A pairs
AMZN_Q4_2024.json - OK | 8059 words | 6 Q&A pairs
AON_Q1_2023.json - OK | 8660 words | 30 Q&A pairs
AON_Q1_2024.json - OK | 7769 words | 15 Q&A pairs
AON_Q2_2023.json - OK | 9749 words | 21 Q&A pairs
AON_Q2_2024.json - OK | 11002 words | 25 Q

In [11]:
processed_files = list(output_dir.glob("*.json"))

total_pairs = 0

for filepath in processed_files:
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)
    total_pairs = total_pairs + data["total_qa_pairs"]

print("Total processed files:", len(processed_files))
print("Total Q&A pairs across all files:", total_pairs)
print("Average Q&A pairs per file:", round(total_pairs / len(processed_files), 1))

Total processed files: 56
Total Q&A pairs across all files: 1082
Average Q&A pairs per file: 19.3


In [3]:
src_dir = Path("raw_transcripts")
output_dir = Path("processed")
src_files = list(src_dir.glob("*.json"))

In [11]:
success_count = 0
skipped_count = 0
failed_count = 0
zero_pairs_files = []

for filepath in src_files:
    output_filename = output_dir / filepath.name
    
    if output_filename.exists():
        skipped_count = skipped_count + 1
        continue
    
    try:
        result = process_one_transcript(filepath)
        
        with open(output_filename, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        print(filepath.name, "- OK |", result["total_word_count"], "words |", result["total_qa_pairs"], "Q&A pairs")
        success_count = success_count + 1
        
        if result["total_qa_pairs"] == 0:
            zero_pairs_files.append(filepath.name)
        
    except Exception as e:
        print(filepath.name, "- FAILED:", str(e))
        failed_count = failed_count + 1

print()
print("Newly processed:", success_count)
print("Skipped (already existed):", skipped_count)
print("Failed:", failed_count)
print("Files with 0 Q&A pairs:", zero_pairs_files)

AAPL_Q1_2021.json - OK | 8309 words | 8 Q&A pairs
AAPL_Q1_2022.json - OK | 7783 words | 18 Q&A pairs
AAPL_Q2_2021.json - OK | 8283 words | 20 Q&A pairs
AAPL_Q2_2022.json - OK | 8151 words | 29 Q&A pairs
AAPL_Q3_2021.json - OK | 8440 words | 28 Q&A pairs
AAPL_Q3_2022.json - OK | 7889 words | 21 Q&A pairs
AAPL_Q4_2021.json - OK | 8448 words | 21 Q&A pairs
AAPL_Q4_2022.json - OK | 7983 words | 19 Q&A pairs
AMZN_Q1_2021.json - OK | 5428 words | 8 Q&A pairs
AMZN_Q1_2022.json - OK | 5626 words | 7 Q&A pairs
AMZN_Q2_2021.json - OK | 6000 words | 8 Q&A pairs
AMZN_Q2_2022.json - OK | 5842 words | 9 Q&A pairs
AMZN_Q3_2021.json - OK | 6836 words | 9 Q&A pairs
AMZN_Q3_2022.json - OK | 5047 words | 7 Q&A pairs
AMZN_Q4_2021.json - OK | 6333 words | 10 Q&A pairs
AMZN_Q4_2022.json - OK | 7264 words | 6 Q&A pairs
AON_Q1_2021.json - OK | 7392 words | 20 Q&A pairs
AON_Q1_2022.json - OK | 5552 words | 9 Q&A pairs
AON_Q2_2021.json - OK | 9036 words | 17 Q&A pairs
AON_Q2_2022.json - OK | 7471 words | 14 Q&A

In [13]:
processed_files = list(output_dir.glob("*.json"))

total_pairs = 0

for filepath in processed_files:
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)
    total_pairs = total_pairs + data["total_qa_pairs"]

print("Total processed files:", len(processed_files))
print("Total Q&A pairs across all files:", total_pairs)
print("Average Q&A pairs per file:", round(total_pairs / len(processed_files), 1))

Total processed files: 160
Total Q&A pairs across all files: 2855
Average Q&A pairs per file: 17.8
